# Chapter 9.4: Embedding Training Optimization

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Implement** lazy embedding updates that only modify accessed embeddings per batch
2. **Design** frequency-based learning rate schedules where popular items get smaller LR
3. **Build** adaptive embedding dimensions that allocate larger dims to frequent features
4. **Compare** embedding initialization strategies (Xavier, pretrained, random) and their impact
5. **Apply** gradient clipping techniques specific to embedding stability
6. **Analyze** the interaction between optimizer choice and embedding convergence
7. **Construct** an optimized embedding training pipeline combining all techniques

## Prerequisites

- PyTorch embedding layers and optimizers
- Understanding of learning rate schedules
- Familiarity with recommendation model training

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/USERNAME/rec_system/blob/main/notebooks/part9/chapter_9.4_embedding_optimization.ipynb)
[![Download](https://img.shields.io/badge/Download-Notebook-blue)](https://github.com/USERNAME/rec_system/blob/main/notebooks/part9/chapter_9.4_embedding_optimization.ipynb)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import time
from typing import List, Dict, Tuple, Optional
from collections import Counter, defaultdict

np.random.seed(42)
torch.manual_seed(42)

print("All imports successful!")
print(f"PyTorch version: {torch.__version__}")

## 1. The Embedding Optimization Challenge

Embedding tables in recommendation systems have unique training challenges:

- **Extreme sparsity**: In a batch of 1024 samples, only ~1000 out of millions of embeddings get gradients
- **Power-law access**: Top 1% of items may account for 50%+ of gradient updates
- **Scale mismatch**: Popular items converge quickly while rare items barely train
- **Memory dominance**: Embedding optimizer state (Adam's m, v) doubles memory for massive tables

### Standard vs. Optimized Training

Standard Adam update for embedding $e_i$:

$$m_i^{(t)} = \beta_1 m_i^{(t-1)} + (1-\beta_1) g_i^{(t)}$$
$$v_i^{(t)} = \beta_2 v_i^{(t-1)} + (1-\beta_2) (g_i^{(t)})^2$$
$$e_i^{(t)} = e_i^{(t-1)} - \frac{\eta}{\sqrt{\hat{v}_i^{(t)}} + \epsilon} \hat{m}_i^{(t)}$$

Problem: For non-accessed embeddings, $g_i = 0$, but Adam still decays $m_i$ and $v_i$ ("phantom updates").

> **💡 Concept:** Sparse optimizers like `SparseAdam` and `Adagrad` are designed to handle this — they only update state for accessed embeddings. Adagrad is the default at Google and Meta for embedding tables because its per-parameter learning rate naturally adapts to frequency.

In [ ]:
# Generate synthetic recommendation data with power-law item distribution
NUM_USERS = 5000
NUM_ITEMS = 50000
EMB_DIM = 32
NUM_INTERACTIONS = 200000

# Zipf distribution for item popularity
item_popularity = np.random.zipf(a=1.5, size=NUM_ITEMS).astype(np.float64)
item_popularity = np.clip(item_popularity, 1, 10000)
item_popularity /= item_popularity.sum()

# Generate interactions
user_ids = np.random.randint(0, NUM_USERS, size=NUM_INTERACTIONS)
item_ids = np.random.choice(NUM_ITEMS, size=NUM_INTERACTIONS, p=item_popularity)

item_counts = Counter(item_ids)

# Analyze distribution
sorted_counts = sorted(item_counts.values(), reverse=True)
top1_pct = sum(sorted_counts[:int(0.01 * NUM_ITEMS)]) / NUM_INTERACTIONS
top10_pct = sum(sorted_counts[:int(0.10 * NUM_ITEMS)]) / NUM_INTERACTIONS

items_no_interaction = NUM_ITEMS - len(item_counts)
print(f"Items with 0 interactions: {items_no_interaction} ({items_no_interaction/NUM_ITEMS:.1%})")
print(f"Top 1% items: {top1_pct:.1%} of interactions")
print(f"Top 10% items: {top10_pct:.1%} of interactions")
print(f"Most popular item: {sorted_counts[0]} interactions ({sorted_counts[0]/NUM_INTERACTIONS:.1%})")

## 2. Lazy Embedding Updates

**Lazy updates** skip optimizer state updates for embeddings not accessed in the current batch. This is critical for:
- **Memory**: Optimizer states (Adam m, v) only need to be stored for active embeddings
- **Compute**: No wasted computation on zero gradients
- **Correctness**: Prevents phantom decay of momentum/variance for idle embeddings

PyTorch's `torch.optim.SparseAdam` implements this natively for sparse gradients.

In [ ]:
class LazyEmbeddingOptimizer:
    """
    Lazy Adam optimizer that only updates accessed embeddings.
    Implements the algorithm from Zinkevich (2003) for sparse online learning.
    """
    
    def __init__(self, embedding: nn.Embedding, lr: float = 0.01,
                 betas: Tuple[float, float] = (0.9, 0.999), eps: float = 1e-8):
        self.embedding = embedding
        self.lr = lr
        self.beta1, self.beta2 = betas
        self.eps = eps
        
        # Lazy state: only allocate for accessed embeddings
        self.m = {}  # First moment
        self.v = {}  # Second moment
        self.step_counts = {}  # Per-embedding step count for bias correction
        self.global_step = 0
        self.update_counts = defaultdict(int)
    
    def step(self, accessed_indices: torch.Tensor):
        """Update only the accessed embeddings."""
        self.global_step += 1
        
        unique_indices = accessed_indices.unique()
        
        with torch.no_grad():
            for idx in unique_indices.tolist():
                grad = self.embedding.weight.grad[idx]
                if grad.abs().sum() == 0:
                    continue
                
                # Initialize state if first access
                if idx not in self.m:
                    self.m[idx] = torch.zeros_like(grad)
                    self.v[idx] = torch.zeros_like(grad)
                    self.step_counts[idx] = 0
                
                self.step_counts[idx] += 1
                self.update_counts[idx] += 1
                t = self.step_counts[idx]
                
                # Adam update
                self.m[idx] = self.beta1 * self.m[idx] + (1 - self.beta1) * grad
                self.v[idx] = self.beta2 * self.v[idx] + (1 - self.beta2) * grad ** 2
                
                # Bias correction
                m_hat = self.m[idx] / (1 - self.beta1 ** t)
                v_hat = self.v[idx] / (1 - self.beta2 ** t)
                
                self.embedding.weight[idx] -= self.lr * m_hat / (v_hat.sqrt() + self.eps)
    
    def state_memory_mb(self) -> float:
        """Return memory used by optimizer states."""
        n_active = len(self.m)
        dim = self.embedding.embedding_dim
        # m + v, each is float32
        return n_active * dim * 2 * 4 / (1024 ** 2)


# Compare lazy vs dense Adam
torch.manual_seed(42)
emb_lazy = nn.Embedding(NUM_ITEMS, EMB_DIM, sparse=True)
nn.init.xavier_uniform_(emb_lazy.weight)
lazy_opt = LazyEmbeddingOptimizer(emb_lazy, lr=0.01)

# Simulate training
for step in range(100):
    batch_items = torch.from_numpy(
        np.random.choice(NUM_ITEMS, size=256, p=item_popularity)
    ).long()
    
    emb_out = emb_lazy(batch_items)
    # Simple loss: push embeddings toward unit norm
    loss = (emb_out.norm(dim=1) - 1.0).pow(2).mean()
    emb_lazy.zero_grad()
    loss.backward()
    lazy_opt.step(batch_items)

# Memory comparison
dense_state_mb = NUM_ITEMS * EMB_DIM * 2 * 4 / (1024 ** 2)  # Full Adam state
lazy_state_mb = lazy_opt.state_memory_mb()

print(f"Active embeddings: {len(lazy_opt.m)} / {NUM_ITEMS} ({len(lazy_opt.m)/NUM_ITEMS:.1%})")
print(f"Dense Adam state memory: {dense_state_mb:.1f} MB")
print(f"Lazy Adam state memory:  {lazy_state_mb:.1f} MB")
print(f"Memory savings: {(1 - lazy_state_mb/dense_state_mb):.1%}")

# Update frequency distribution
update_freqs = sorted(lazy_opt.update_counts.values(), reverse=True)
print(f"\nMax updates for single embedding: {update_freqs[0]}")
print(f"Median updates: {np.median(update_freqs):.0f}")

## 3. Frequency-Based Learning Rates

Popular items receive many gradient updates, leading to:
- **Over-fitting** on popular items
- **Under-fitting** on rare items

Solution: Scale the learning rate inversely with item frequency:

$$\eta_i = \eta_0 \cdot \left(\frac{f_i}{f_{\max}}\right)^{-\gamma}$$

where $f_i$ is the access frequency of item $i$ and $\gamma \in [0, 1]$ controls the scaling strength.

Alternatively, use **Adagrad-style** per-parameter learning rates:

$$\eta_i^{(t)} = \frac{\eta_0}{\sqrt{\sum_{\tau=1}^{t} g_{i,\tau}^2} + \epsilon}$$

This naturally reduces LR for frequently-updated embeddings.

> **🔑 Pro Tip:** Adagrad is the default optimizer for embedding tables at Google (used in TF Serving) and Meta (used in DLRM). Its natural frequency-based LR adaptation is a key reason, despite its theoretical convergence to zero LR over time.

In [ ]:
class FrequencyAwareLRScheduler:
    """
    Adjusts learning rate per embedding based on access frequency.
    Inspired by practices at Google (2016) and Meta (2019).
    """
    
    def __init__(self, num_items: int, base_lr: float = 0.01,
                 gamma: float = 0.5, warmup_steps: int = 100):
        self.num_items = num_items
        self.base_lr = base_lr
        self.gamma = gamma
        self.warmup_steps = warmup_steps
        
        self.access_counts = np.zeros(num_items)
        self.global_step = 0
    
    def get_lr(self, item_ids: np.ndarray) -> np.ndarray:
        """Get per-item learning rates."""
        self.global_step += 1
        
        # Update access counts
        for idx in item_ids:
            self.access_counts[idx] += 1
        
        # Compute per-item LR
        counts = self.access_counts[item_ids]
        max_count = max(self.access_counts.max(), 1)
        
        # Frequency scaling: popular items get lower LR
        freq_factor = (counts / max_count + 0.01) ** (-self.gamma)
        freq_factor = np.clip(freq_factor, 0.1, 10.0)
        
        # Warmup
        warmup_factor = min(self.global_step / self.warmup_steps, 1.0)
        
        return self.base_lr * freq_factor * warmup_factor


# Demonstrate LR adaptation
scheduler = FrequencyAwareLRScheduler(NUM_ITEMS, base_lr=0.01, gamma=0.5)

# Simulate 500 batches
lr_history = defaultdict(list)
tracked_items = [0, 100, 1000, 10000, 40000]  # different popularity tiers

for step in range(500):
    batch = np.random.choice(NUM_ITEMS, size=256, p=item_popularity)
    lrs = scheduler.get_lr(batch)
    
    for item_id in tracked_items:
        if item_id in batch:
            idx = np.where(batch == item_id)[0][0]
            lr_history[item_id].append((step, lrs[idx]))

# Print final access counts for tracked items
for item_id in tracked_items:
    count = int(scheduler.access_counts[item_id])
    n_lr_records = len(lr_history[item_id])
    avg_lr = np.mean([lr for _, lr in lr_history[item_id]]) if lr_history[item_id] else 0
    print(f"Item {item_id:>5}: count={count:>4}, avg_lr={avg_lr:.6f}")

In [ ]:
# Visualize frequency-based LR adaptation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# LR over time for different items
colors = plt.cm.viridis(np.linspace(0, 0.9, len(tracked_items)))
for color, item_id in zip(colors, tracked_items):
    if lr_history[item_id]:
        steps, lrs_vals = zip(*lr_history[item_id])
        count = int(scheduler.access_counts[item_id])
        axes[0].scatter(steps, lrs_vals, s=8, alpha=0.5, color=color, 
                       label=f'Item {item_id} (count={count})')

axes[0].set_xlabel('Training Step', fontsize=12)
axes[0].set_ylabel('Learning Rate', fontsize=12)
axes[0].set_title('Per-Item Learning Rate Over Time', fontsize=13)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# LR vs frequency
all_counts = scheduler.access_counts[scheduler.access_counts > 0]
max_count = all_counts.max()
freq_range = np.linspace(1, max_count, 1000)

for gamma_val in [0.0, 0.25, 0.5, 0.75, 1.0]:
    lr_vals = 0.01 * (freq_range / max_count + 0.01) ** (-gamma_val)
    lr_vals = np.clip(lr_vals, 0.001, 0.1)
    axes[1].plot(freq_range, lr_vals, label=f'γ={gamma_val}', linewidth=2)

axes[1].set_xlabel('Access Count', fontsize=12)
axes[1].set_ylabel('Learning Rate', fontsize=12)
axes[1].set_title('LR vs Frequency for Different γ', fontsize=13)
axes[1].set_xscale('log')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Adaptive Embedding Dimensions

**Idea**: Not all items need the same embedding dimension. Frequent items benefit from higher capacity (larger dim), while rare items are better served by smaller dims to avoid overfitting.

This approach was formalized by Ginart et al. (2021, Stanford) as **Mixed Dimension Embeddings** and used in practice by Facebook in AutoDim (Zhao et al., 2021).

$$d_i = d_{\min} + \lfloor (d_{\max} - d_{\min}) \cdot \left(\frac{\log(f_i + 1)}{\log(f_{\max} + 1)}\right)^\alpha \rfloor$$

where $d_i$ is the dimension for item $i$, $f_i$ is its frequency, and $\alpha$ controls the scaling.

> **⚠️ Common Pitfall:** Variable-dimension embeddings require padding or projection to a common dimension before feeding into the MLP. The projection layer must be learned, adding some overhead.

In [ ]:
class AdaptiveDimensionEmbedding(nn.Module):
    """
    Embedding table with adaptive dimensions based on feature frequency.
    Inspired by Ginart et al. (2021) and Facebook's AutoDim (2021).
    """
    
    def __init__(self, num_items: int, output_dim: int, item_counts: Dict[int, int],
                 dim_min: int = 4, dim_max: int = 64, alpha: float = 0.5):
        super().__init__()
        self.output_dim = output_dim
        self.num_items = num_items
        
        # Determine dimension for each item based on frequency
        counts = np.array([item_counts.get(i, 0) for i in range(num_items)], dtype=np.float64)
        max_count = max(counts.max(), 1)
        
        # Compute per-item dimensions
        log_freq = np.log(counts + 1) / np.log(max_count + 1)
        dims = dim_min + np.floor((dim_max - dim_min) * log_freq ** alpha).astype(int)
        dims = np.clip(dims, dim_min, dim_max)
        
        # Group items by dimension for efficient storage
        self.dim_groups = defaultdict(list)
        self.item_to_dim = {}
        for i, d in enumerate(dims):
            self.dim_groups[d].append(i)
            self.item_to_dim[i] = d
        
        # Create embedding tables for each dimension group
        self.embeddings = nn.ModuleDict()
        self.projections = nn.ModuleDict()
        self.item_to_local_idx = {}
        
        for dim, items in self.dim_groups.items():
            self.embeddings[str(dim)] = nn.Embedding(len(items), dim)
            nn.init.xavier_uniform_(self.embeddings[str(dim)].weight)
            if dim != output_dim:
                self.projections[str(dim)] = nn.Linear(dim, output_dim, bias=False)
            for local_idx, global_idx in enumerate(items):
                self.item_to_local_idx[global_idx] = (str(dim), local_idx)
        
        self.dims_array = dims
    
    def forward(self, item_ids: torch.Tensor) -> torch.Tensor:
        """Look up and project embeddings to common dimension."""
        batch_size = item_ids.size(0)
        output = torch.zeros(batch_size, self.output_dim)
        
        for i, idx in enumerate(item_ids.tolist()):
            dim_key, local_idx = self.item_to_local_idx[idx]
            emb = self.embeddings[dim_key](torch.tensor([local_idx]))
            if dim_key in self.projections:
                emb = self.projections[dim_key](emb)
            output[i] = emb.squeeze(0)
        
        return output
    
    def total_params(self) -> int:
        return sum(p.numel() for p in self.parameters())
    
    def memory_mb(self) -> float:
        return self.total_params() * 4 / (1024 ** 2)


# Compare fixed vs adaptive dimensions
adaptive_emb = AdaptiveDimensionEmbedding(
    NUM_ITEMS, output_dim=32, item_counts=item_counts,
    dim_min=4, dim_max=64, alpha=0.5
)

fixed_emb = nn.Embedding(NUM_ITEMS, 32)

print(f"Fixed Embedding:    {sum(p.numel() for p in fixed_emb.parameters()):>12,} params, "
      f"{sum(p.numel() for p in fixed_emb.parameters()) * 4 / (1024**2):.1f} MB")
print(f"Adaptive Embedding: {adaptive_emb.total_params():>12,} params, "
      f"{adaptive_emb.memory_mb():.1f} MB")

# Distribution of dimensions
dim_dist = Counter(adaptive_emb.dims_array)
print(f"\nDimension distribution:")
for dim, count in sorted(dim_dist.items()):
    print(f"  dim={dim:>2}: {count:>6} items ({count/NUM_ITEMS:.1%})")

## 5. Embedding Initialization Strategies

Initialization significantly affects convergence speed and final quality:

| Strategy | Formula | Best For |
|----------|---------|----------|
| **Xavier/Glorot** | $\mathcal{N}(0, \frac{2}{d_{in}+d_{out}})$ | General purpose |
| **He/Kaiming** | $\mathcal{N}(0, \frac{2}{d_{in}})$ | ReLU networks |
| **Truncated Normal** | $\mathcal{N}(0, \frac{1}{\sqrt{d}})$ clipped to $\pm 2\sigma$ | Google's default |
| **Pretrained (warm start)** | From previous model | Incremental retraining |
| **Frequency-scaled** | $\mathcal{N}(0, \frac{c}{\sqrt{f_i \cdot d}})$ | Frequency-aware |

> **🔑 Pro Tip:** Google's recommendation is to use truncated normal with $\sigma = 1/\sqrt{d}$ for embedding tables. Meta uses Xavier uniform. The key insight is that the initial embedding norm should be similar across items regardless of their frequency.

In [ ]:
class EmbeddingInitializer:
    """Various initialization strategies for embedding tables."""
    
    @staticmethod
    def xavier_uniform(emb: nn.Embedding):
        nn.init.xavier_uniform_(emb.weight)
    
    @staticmethod
    def xavier_normal(emb: nn.Embedding):
        nn.init.xavier_normal_(emb.weight)
    
    @staticmethod
    def truncated_normal(emb: nn.Embedding, std: Optional[float] = None):
        d = emb.embedding_dim
        if std is None:
            std = 1.0 / np.sqrt(d)
        with torch.no_grad():
            emb.weight.normal_(0, std)
            # Truncate at 2*std
            emb.weight.clamp_(-2 * std, 2 * std)
    
    @staticmethod
    def frequency_scaled(emb: nn.Embedding, item_counts: Dict[int, int],
                         base_std: float = 0.01):
        d = emb.embedding_dim
        with torch.no_grad():
            for i in range(emb.num_embeddings):
                freq = item_counts.get(i, 1)
                std = base_std / np.sqrt(freq * d)
                emb.weight[i].normal_(0, std)
    
    @staticmethod
    def uniform_unit_norm(emb: nn.Embedding):
        """Initialize on the unit hypersphere."""
        with torch.no_grad():
            emb.weight.normal_(0, 1)
            norms = emb.weight.norm(dim=1, keepdim=True)
            emb.weight.div_(norms)


# Compare initialization strategies on a training task
def train_with_init(init_fn, init_name, n_items=5000, emb_dim=32, n_steps=300):
    torch.manual_seed(42)
    np.random.seed(42)
    
    user_emb = nn.Embedding(1000, emb_dim)
    item_emb = nn.Embedding(n_items, emb_dim)
    
    nn.init.xavier_uniform_(user_emb.weight)
    init_fn(item_emb)
    
    optimizer = torch.optim.Adam(list(user_emb.parameters()) + list(item_emb.parameters()), lr=0.005)
    
    pop = np.random.zipf(1.5, n_items).astype(np.float64)
    pop = np.clip(pop, 1, 1000)
    pop /= pop.sum()
    
    losses = []
    for step in range(n_steps):
        users = torch.randint(0, 1000, (256,))
        pos_items = torch.from_numpy(np.random.choice(n_items, 256, p=pop)).long()
        neg_items = torch.randint(0, n_items, (256,))
        
        u = user_emb(users)
        pos = item_emb(pos_items)
        neg = item_emb(neg_items)
        
        pos_score = (u * pos).sum(dim=1)
        neg_score = (u * neg).sum(dim=1)
        
        loss = F.softplus(neg_score - pos_score).mean()  # BPR loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    
    return losses


init_strategies = {
    'Xavier Uniform': lambda emb: EmbeddingInitializer.xavier_uniform(emb),
    'Xavier Normal': lambda emb: EmbeddingInitializer.xavier_normal(emb),
    'Truncated Normal': lambda emb: EmbeddingInitializer.truncated_normal(emb),
    'Unit Norm': lambda emb: EmbeddingInitializer.uniform_unit_norm(emb),
}

all_losses = {}
for name, init_fn in init_strategies.items():
    all_losses[name] = train_with_init(init_fn, name)
    print(f"{name}: final loss = {np.mean(all_losses[name][-20:]):.4f}")

In [ ]:
# Visualize initialization impact
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training curves
window = 10
smooth = lambda x: np.convolve(x, np.ones(window)/window, mode='valid')
colors = ['steelblue', 'coral', 'green', 'purple']

for (name, losses), color in zip(all_losses.items(), colors):
    axes[0].plot(smooth(losses), label=name, color=color, linewidth=2)

axes[0].set_xlabel('Training Step', fontsize=12)
axes[0].set_ylabel('BPR Loss (smoothed)', fontsize=12)
axes[0].set_title('Convergence by Initialization Strategy', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Initial weight norm distributions
for (name, init_fn), color in zip(init_strategies.items(), colors):
    torch.manual_seed(42)
    test_emb = nn.Embedding(5000, 32)
    init_fn(test_emb)
    norms = test_emb.weight.norm(dim=1).detach().numpy()
    axes[1].hist(norms, bins=50, alpha=0.5, label=name, color=color, density=True)

axes[1].set_xlabel('Embedding L2 Norm', fontsize=12)
axes[1].set_ylabel('Density', fontsize=12)
axes[1].set_title('Initial Embedding Norm Distribution', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Gradient Clipping for Embedding Stability

Embedding gradients can be noisy and have high variance, especially for:
- Newly added items with few interactions
- Items in a "cold start" phase
- Features that appear in unusually large batches

### Clipping Strategies

1. **Global norm clipping**: $g \leftarrow g \cdot \min(1, \frac{c}{\|g\|_2})$
2. **Per-row clipping**: Clip each embedding row's gradient independently
3. **Adaptive clipping**: Scale clip threshold by running average gradient norm

> **⚠️ Common Pitfall:** Global gradient clipping with embeddings can be misleading — if you have millions of embeddings but only 1000 are active, the global norm is dominated by a few hot embeddings. Per-row clipping is often more appropriate.

In [ ]:
class EmbeddingGradientClipper:
    """Gradient clipping strategies for embedding tables."""
    
    @staticmethod
    def global_norm_clip(embedding: nn.Embedding, max_norm: float):
        """Standard global norm clipping."""
        if embedding.weight.grad is not None:
            torch.nn.utils.clip_grad_norm_([embedding.weight], max_norm)
    
    @staticmethod
    def per_row_clip(embedding: nn.Embedding, max_norm: float, accessed_indices: torch.Tensor):
        """Clip each active embedding row independently."""
        if embedding.weight.grad is None:
            return
        with torch.no_grad():
            unique_idx = accessed_indices.unique()
            for idx in unique_idx:
                row_grad = embedding.weight.grad[idx]
                row_norm = row_grad.norm()
                if row_norm > max_norm:
                    embedding.weight.grad[idx] = row_grad * (max_norm / row_norm)
    
    @staticmethod
    def adaptive_clip(embedding: nn.Embedding, accessed_indices: torch.Tensor,
                      running_norm: Dict, alpha: float = 0.99, multiplier: float = 3.0):
        """Clip based on running average gradient norm."""
        if embedding.weight.grad is None:
            return
        with torch.no_grad():
            unique_idx = accessed_indices.unique()
            for idx in unique_idx.tolist():
                row_grad = embedding.weight.grad[idx]
                row_norm = row_grad.norm().item()
                
                if idx not in running_norm:
                    running_norm[idx] = row_norm
                else:
                    running_norm[idx] = alpha * running_norm[idx] + (1 - alpha) * row_norm
                
                max_norm = multiplier * running_norm[idx]
                if row_norm > max_norm and max_norm > 0:
                    embedding.weight.grad[idx] = row_grad * (max_norm / row_norm)


# Demonstrate clipping effects
def train_with_clipping(clip_strategy: str, n_steps: int = 300):
    torch.manual_seed(42)
    np.random.seed(42)
    
    emb = nn.Embedding(5000, 32)
    nn.init.xavier_uniform_(emb.weight)
    optimizer = torch.optim.SGD(emb.parameters(), lr=0.1)
    running_norm = {}
    
    losses = []
    grad_norms = []
    
    pop = np.random.zipf(1.5, 5000).astype(np.float64)
    pop = np.clip(pop, 1, 1000)
    pop /= pop.sum()
    
    for step in range(n_steps):
        items = torch.from_numpy(np.random.choice(5000, 256, p=pop)).long()
        out = emb(items)
        
        # Synthetic loss with occasional gradient spikes
        target = torch.randn_like(out)
        if step % 50 == 0:  # Simulate gradient spike
            target *= 100
        loss = (out - target).pow(2).mean()
        
        optimizer.zero_grad()
        loss.backward()
        
        # Record gradient norm before clipping
        grad_norm = emb.weight.grad[items.unique()].norm().item()
        
        # Apply clipping
        clipper = EmbeddingGradientClipper()
        if clip_strategy == 'none':
            pass
        elif clip_strategy == 'global':
            clipper.global_norm_clip(emb, max_norm=1.0)
        elif clip_strategy == 'per_row':
            clipper.per_row_clip(emb, max_norm=0.5, accessed_indices=items)
        elif clip_strategy == 'adaptive':
            clipper.adaptive_clip(emb, items, running_norm)
        
        optimizer.step()
        losses.append(loss.item())
        grad_norms.append(grad_norm)
    
    return losses, grad_norms


strategies = ['none', 'global', 'per_row', 'adaptive']
results = {}
for strategy in strategies:
    results[strategy] = train_with_clipping(strategy)
    final_loss = np.mean(results[strategy][0][-20:])
    print(f"{strategy:>10}: final_loss={final_loss:.4f}")

In [ ]:
# Visualize gradient clipping effects
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {'none': 'gray', 'global': 'steelblue', 'per_row': 'coral', 'adaptive': 'green'}

for strategy in strategies:
    losses, gnorms = results[strategy]
    axes[0].plot(smooth(losses), label=strategy, color=colors[strategy], linewidth=2, alpha=0.8)

axes[0].set_xlabel('Step', fontsize=12)
axes[0].set_ylabel('Loss (smoothed)', fontsize=12)
axes[0].set_title('Training Loss by Clipping Strategy', fontsize=13)
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_yscale('log')

for strategy in strategies:
    _, gnorms = results[strategy]
    axes[1].plot(gnorms, label=strategy, color=colors[strategy], linewidth=1, alpha=0.7)

axes[1].set_xlabel('Step', fontsize=12)
axes[1].set_ylabel('Gradient Norm', fontsize=12)
axes[1].set_title('Gradient Norm by Clipping Strategy', fontsize=13)
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

## 🏋️ Exercise 1: Implement Frequency-Aware Learning Rate Scheduling

Build a custom optimizer that combines Adagrad-style per-parameter learning rates with frequency-aware scaling.

In [ ]:
# 🏋️ Exercise 1: Frequency-aware Adagrad

class FrequencyAwareAdagrad:
    def __init__(self, embedding: nn.Embedding, base_lr: float = 0.1,
                 gamma: float = 0.5, eps: float = 1e-10):
        """
        Adagrad optimizer with additional frequency-based LR scaling.
        
        lr_i = base_lr * (count_i / max_count) ^ (-gamma) / sqrt(G_i + eps)
        
        Args:
            embedding: The embedding layer to optimize
            base_lr: Base learning rate
            gamma: Frequency scaling exponent
            eps: Small constant for numerical stability
        """
        # TODO: Initialize
        # - Sum of squared gradients per embedding (Adagrad state)
        # - Access counts per embedding
        pass
    
    def step(self, accessed_indices: torch.Tensor):
        """
        Update accessed embeddings.
        
        TODO:
        1. Update access counts for accessed indices
        2. Accumulate squared gradients
        3. Compute frequency-scaled LR
        4. Apply Adagrad update with scaled LR
        """
        pass
    
    def get_effective_lr(self, item_id: int) -> float:
        """Return the current effective LR for a given item."""
        # TODO
        pass


# Test:
# emb = nn.Embedding(5000, 32)
# opt = FrequencyAwareAdagrad(emb, base_lr=0.1, gamma=0.5)
# for _ in range(100):
#     ids = torch.randint(0, 5000, (256,))
#     out = emb(ids)
#     loss = out.pow(2).mean()
#     emb.zero_grad()
#     loss.backward()
#     opt.step(ids)
# print(f"Effective LR for item 0: {opt.get_effective_lr(0):.6f}")
# print(f"Effective LR for item 4999: {opt.get_effective_lr(4999):.6f}")

## 🏋️ Exercise 2: Build an Optimized Embedding Training Pipeline

Combine all techniques: lazy updates, frequency-aware LR, adaptive clipping, and smart initialization into a single pipeline.

In [ ]:
# 🏋️ Exercise 2: Combined optimization pipeline

class OptimizedEmbeddingPipeline:
    def __init__(self, num_items: int, embedding_dim: int, item_counts: Dict[int, int],
                 base_lr: float = 0.01, gamma: float = 0.5, clip_multiplier: float = 3.0):
        """
        Fully optimized embedding training pipeline.
        
        TODO:
        1. Initialize embedding with truncated normal
        2. Set up lazy Adam optimizer
        3. Configure frequency-aware LR scheduling
        4. Set up adaptive gradient clipping
        """
        pass
    
    def train_step(self, item_ids: torch.Tensor, targets: torch.Tensor) -> Dict:
        """
        Execute one optimized training step.
        
        Returns dict with loss, grad_norm, effective_lr, clipped_count
        """
        # TODO:
        # 1. Forward pass
        # 2. Loss computation
        # 3. Backward pass
        # 4. Adaptive gradient clipping
        # 5. Frequency-aware LR adjustment
        # 6. Lazy optimizer step
        pass


# Test:
# pipeline = OptimizedEmbeddingPipeline(5000, 32, item_counts)
# for step in range(100):
#     ids = torch.from_numpy(np.random.choice(5000, 256, p=item_popularity[:5000]/item_popularity[:5000].sum())).long()
#     targets = torch.randn(256, 32)
#     result = pipeline.train_step(ids, targets)
#     if step % 25 == 0:
#         print(f"Step {step}: {result}")

## 🏋️ Exercise 3: Compare Optimizer Choices for Embeddings

Run a systematic comparison of SGD, Adam, SparseAdam, and Adagrad on an embedding training task.

In [ ]:
# 🏋️ Exercise 3: Optimizer comparison

def compare_optimizers(n_items: int = 5000, emb_dim: int = 32, n_steps: int = 500) -> Dict:
    """
    Compare different optimizers for embedding training.
    
    TODO:
    1. Create identical initial embeddings for fair comparison
    2. Train with SGD (lr=0.1), Adam (lr=0.001), SparseAdam (lr=0.01), Adagrad (lr=0.1)
    3. Use BPR loss with zipf-distributed item sampling
    4. Track:
       - Training loss curves
       - Embedding norm evolution
       - Per-item convergence (popular vs rare)
    5. Return comparison results
    """
    pass


# results = compare_optimizers()
# Plot the comparison

## Summary

| Technique | Benefit | When to Use |
|-----------|---------|-------------|
| **Lazy Updates** | 50-99% memory savings on optimizer state | Always, for large embedding tables |
| **Frequency-Aware LR** | Better balance between popular/rare items | When popularity skew is extreme |
| **Adaptive Dimensions** | 30-60% memory savings with minimal quality loss | When memory is a constraint |
| **Truncated Normal Init** | Faster convergence, more stable training | Default recommendation |
| **Per-Row Gradient Clipping** | Prevents training instability from gradient spikes | Always recommended |
| **Adagrad Optimizer** | Natural frequency adaptation for embeddings | Production default at Google/Meta |

### Key References
- Duchi et al., "Adaptive Subgradient Methods for Online Learning" (2011 — Adagrad)
- Ginart et al., "Mixed Dimension Embeddings with Application to Memory-Efficient Recommendation Systems" (2021, Stanford)
- Zhao et al., "AutoDim: Field-aware Embedding Dimension Searchin Recommender Systems" (2021, Meta)
- Naumov et al., "Deep Learning Recommendation Model for Personalization and Recommendation Systems" (2019, Meta — DLRM)

### Next Steps
In the next notebook (9.5), we will explore incremental and online training approaches, including drift detection and staleness management.